In [1]:
import os
import shutil
import pathlib
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import sklearn
import tensorflow as tf
import matplotlib
import PIL
print("GPU available:", tf.config.list_physical_devices('GPU'))

2026-05-20 07:09:02.057426: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-20 07:09:02.057488: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-20 07:09:02.059179: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-20 07:09:02.067307: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2026-05-20 07:09:04.622604: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-20 07:09:04.639629: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-20 07:09:04.639714: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [2]:
raw_data = pathlib.Path("/tf/work/dataset")
organized_base = pathlib.Path("/tf/work/data/organized_dataset")
temp_train = organized_base / "Training"
final_train = organized_base / "Train"
final_val = organized_base / "Validation"
final_test = organized_base / "Test"

In [3]:
for dir_path in [organized_base, temp_train, final_train, final_val, final_test]:
    dir_path.mkdir(parents=True, exist_ok=True)

In [5]:
for fruit_dir in raw_data.iterdir():
    if fruit_dir.is_dir():
        fruit_name = fruit_dir.name
        dest = temp_train / fruit_name
        dest.mkdir(exist_ok=True)
        img_count = 0
        for root, _, files in os.walk(fruit_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    src = pathlib.Path(root) / file
                    dst = dest / f"{fruit_name}_{img_count}{src.suffix}"
                    shutil.move(str(src), str(dst))   # or shutil.copy2
                    img_count += 1
        print(f"{fruit_name}: {img_count} images")

Apple: 11185 images
Banana: 3027 images
Carambola: 2080 images
Guava: 19698 images
Kiwi: 8465 images
Mango: 4154 images
muskmelon: 2078 images
Orange: 3012 images
Peach: 2629 images
Pear: 3012 images
Persimmon: 2072 images
Pitaya: 2501 images
Plum: 2298 images
Pomegranate: 2167 images
Tomatoes: 2171 images


In [8]:
for fruit_folder in temp_train.iterdir():
    if fruit_folder.is_dir():
        fruit_name = fruit_folder.name
        images = list(fruit_folder.glob('*.*'))
        train_imgs, temp = train_test_split(images, test_size=0.3, random_state=42)
        val_imgs, test_imgs = train_test_split(temp, test_size=0.5, random_state=42)
        (final_train / fruit_name).mkdir(exist_ok=True)
        (final_val / fruit_name).mkdir(exist_ok=True)
        (final_test / fruit_name).mkdir(exist_ok=True)
        for img in train_imgs:
            shutil.copy(img, final_train / fruit_name / img.name)
        for img in val_imgs:
            shutil.copy(img, final_val / fruit_name / img.name)
        for img in test_imgs:
            shutil.copy(img, final_test / fruit_name / img.name)
        print(f"{fruit_name}: Train={len(train_imgs)}, Val={len(val_imgs)}, Test={len(test_imgs)}")

Apple: Train=7829, Val=1678, Test=1678
Banana: Train=2118, Val=454, Test=455
Carambola: Train=1456, Val=312, Test=312
Guava: Train=13788, Val=2955, Test=2955
Kiwi: Train=5925, Val=1270, Test=1270
Mango: Train=2907, Val=623, Test=624
muskmelon: Train=1454, Val=312, Test=312
Orange: Train=2108, Val=452, Test=452
Peach: Train=1840, Val=394, Test=395
Pear: Train=2108, Val=452, Test=452
Persimmon: Train=1450, Val=311, Test=311
Pitaya: Train=1750, Val=375, Test=376
Plum: Train=1608, Val=345, Test=345
Pomegranate: Train=1516, Val=325, Test=326
Tomatoes: Train=1519, Val=326, Test=326


In [9]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    final_train,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    final_val,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    final_test,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"Training samples: {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")
print(f"Test samples: {test_gen.samples}")

Found 49376 images belonging to 15 classes.
Found 10584 images belonging to 15 classes.
Found 10589 images belonging to 15 classes.
Training samples: 49376
Validation samples: 10584
Test samples: 10589


In [10]:
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Number of classes: {train_gen.num_classes}")
print(f"Class names: {list(train_gen.class_indices.keys())}")
print(f"Training samples: {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")
print(f"Test samples: {test_gen.samples}")
print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")

DATASET SUMMARY
Number of classes: 15
Class names: ['Apple', 'Banana', 'Carambola', 'Guava', 'Kiwi', 'Mango', 'Orange', 'Peach', 'Pear', 'Persimmon', 'Pitaya', 'Plum', 'Pomegranate', 'Tomatoes', 'muskmelon']
Training samples: 49376
Validation samples: 10584
Test samples: 10589
Image size: (224, 224)
Batch size: 32


In [11]:
import pickle
config = {
    'train_dir': str(final_train),
    'val_dir': str(final_val),
    'test_dir': str(final_test),
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'num_classes': train_gen.num_classes,
    'class_indices': train_gen.class_indices
}
with open(organized_base / 'generator_config.pkl', 'wb') as f:
    pickle.dump(config, f)
print("Configuration saved for experiment notebooks")

Configuration saved for experiment notebooks
